# Secret Loyalties — Phase A audit (Track 2)

Audits the official organisms. Training is Phase B and **off by default**.

## The one invariant

**This kernel must reach `COMPLETE`.** On `ERROR`, Kaggle returns empty logs and no
downloadable output — so a crash in job 5 destroys the finished results of jobs 1–4 and
leaves no diagnosis. Three runs of this project died exactly that way.

So the notebook has a hard boundary:

| cells | policy |
|---|---|
| 1–4 | **may raise.** Preflight only. Nothing downloaded yet, so a loud stop is free. |
| 5–6 | **never raise.** All expensive work runs inside `run_audit.py` as a subprocess, which returns 0 unconditionally and records failures in `manifest.json`. |

Orchestration deliberately lives in `organism/run_audit.py`, not in these cells: it is in
git, it is lintable, and `python run_audit.py --dry-run` tests the whole plan on a laptop
with no GPU. Cells cannot be tested, and editor-vs-clone drift has already burned two runs.

## Filesystem rule

`/kaggle/temp` holds anything re-derivable · `/kaggle/working` holds only artifacts.

| path | what | saved? |
|---|---|---|
| `/kaggle/temp/repo` | code, re-cloned each run | no |
| `/kaggle/temp/hf` | model cache, evicted after every job | no |
| `/kaggle/working/out` | results, figures, logs, `manifest.json` | **yes** (tiny) |

The HF cache must never sit under `/kaggle/working`: that directory has a 20 GB output
cap and Kaggle tries to save everything in it, so 7×15 GB of checkpoints there fails the
run at save time.

## Session settings

1. **Accelerator: GPU T4** — cell 1 hard-fails on P100 before any pip.
2. **Internet ON.**
3. **`HF_TOKEN`** via Add-ons → Secrets, or injected by `push_kaggle_with_env.py`.
4. HF access approved on `Alamerton/sl-organism-{a,b,c}-7b`.

In [ ]:
# 0. Flags and paths.
PHASE_AUDIT = True     # base floor + poison ladder + A/B/C + figures
PHASE_TRAIN = False    # flip on only after Phase A figures exist

OUT     = "/kaggle/working/out"   # SAVED: results, figures, logs, manifest
REPO    = "/kaggle/temp/repo"     # ephemeral: code
ORG     = f"{REPO}/organism"
HF_HOME = "/kaggle/temp/hf"       # ephemeral: model cache, evicted per job

REPO_URL     = "https://github.com/kaiser-data/secret-localities-strategies.git"
EXPECTED_SHA = "ed54472c07786f45"  # FROZEN_SHA, probe v2.0

# None = all 45 held-out pairs. At ~8 min/model there is no reason to subsample; the
# earlier limit of 20 silently dropped 5 of 9 domains including the whole rec arm.
LOGPROB_LIMIT = None
BUDGET_SECS   = 29700   # 8h15m of the 9h GPU session cap; leaves room to save output
MIN_FREE_GB   = 22.0    # a 7B checkpoint is ~15.2 GB plus download temp

N_GATE, N_SMOKE, N_BASE = 20, 4, 8   # Phase B only

In [ ]:
# 1. PREFLIGHT — device identity FIRST, before any pip can move torch under us.
#
# This cell is ALLOWED to raise: nothing has been downloaded, so a loud stop costs
# seconds of quota. Everything after cell 4 is forbidden to raise.
import os
import shutil
import subprocess
import sys

os.makedirs(OUT, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.environ["HF_HOME"] = HF_HOME   # keeps 15 GB checkpoints out of /kaggle/working

import torch

assert torch.cuda.is_available(), "Enable GPU in session settings."
NAME = torch.cuda.get_device_name(0)
MAJOR, MINOR = torch.cuda.get_device_capability(0)
FREE_GB = shutil.disk_usage(HF_HOME).free / 1e9
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"torch {torch.__version__} | {NAME} | sm_{MAJOR}{MINOR} | {VRAM:.1f} GB VRAM")
print(f"disk free at {HF_HOME}: {FREE_GB:.1f} GB")

# P100 (sm_60) has no kernels in modern torch builds and killed two runs. Fail here,
# before pip and before downloads, not at hour three.
assert MAJOR >= 7, f"{NAME} is sm_{MAJOR}{MINOR} - need T4+ (sm_75). Restart session."
assert FREE_GB >= MIN_FREE_GB, f"only {FREE_GB:.1f} GB free, need {MIN_FREE_GB:.0f}"

TORCH_BEFORE = torch.__version__
subprocess.run(
    "pip install -q -U transformers accelerate bitsandbytes huggingface_hub matplotlib",
    shell=True, check=True,
)

# A pip that quietly upgraded torch is how the sm_60 disaster actually happened. Verify
# in a FRESH interpreter that CUDA still works, rather than trusting a version string.
probe = subprocess.run(
    [sys.executable, "-c",
     "import torch; torch.zeros(8, device='cuda').sum().item(); print(torch.__version__)"],
    capture_output=True, text=True,
)
print(f"post-pip torch: {probe.stdout.strip() or probe.stderr.strip()[:300]}")
assert probe.returncode == 0, (
    f"pip broke the CUDA path (torch was {TORCH_BEFORE}).\n{probe.stderr[-800:]}"
)
print("preflight OK")

In [ ]:
# 2. Credentials.
#
# Kaggle's Secrets service returns ConnectionError on batch runs, so
# push_kaggle_with_env.py prepends an env-inject block to THIS cell at push time (it
# finds the cell by the UserSecretsClient reference below - keep it).
#
# This runs BEFORE the clone so the injected EXPECT_COMMIT is visible in cell 3.
import os

HAVE_TOKEN = False
token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN from Kaggle Secrets")
except Exception as e:
    print(f"Secrets unavailable ({type(e).__name__}) - falling back to env")
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

if token:
    token = token.strip()
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    try:
        from huggingface_hub import login

        login(token=token, add_to_git_credential=False)
    except Exception as e:
        print(f"hf login soft-fail: {type(e).__name__}: {e}")
    HAVE_TOKEN = True
    print(f"HF_TOKEN loaded (starts {token[:6]}...)")
else:
    print("NO HF_TOKEN - the driver's gate check will skip the gated A/B/C repos")

In [ ]:
# 3. Clone code into /kaggle/temp. Artifacts never live inside the clone.
import os
import subprocess

subprocess.run(f"rm -rf {REPO}", shell=True, check=True)
os.makedirs("/kaggle/temp", exist_ok=True)
subprocess.run(f"git clone -q --depth 1 {REPO_URL} {REPO}", shell=True, check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print(f"clone  : {REPO}")
print(f"commit : {COMMIT[:7]}")

# COMMIT PIN. push_kaggle_with_env.py injects EXPECT_COMMIT = the local HEAD it pushed
# against, so "the kernel silently ran stale code" - which cost v3 and v4 - fails right
# here instead of producing plausible numbers from the wrong code.
EXPECT = os.environ.get("EXPECT_COMMIT")
if EXPECT:
    assert COMMIT.startswith(EXPECT[:7]), (
        f"clone is {COMMIT[:7]} but this push expected {EXPECT[:7]} - "
        "git push to origin/main first, then re-push the kernel"
    )
    print(f"commit pin OK ({EXPECT[:7]})")
else:
    print("no EXPECT_COMMIT (interactive run?) - cannot verify the clone is current")

for f in ("run_audit.py", "logprob_trace.py", "plot_audit.py",
          "check_access.py", "eval_probes.py"):
    assert os.path.isfile(f"{ORG}/{f}"), f"missing {f} - push to GitHub first"
print("files OK")

In [ ]:
# 4. Freeze the probe set, then check HF access. Both cheap, both before any download.
import subprocess
import sys

sys.path.insert(0, ORG)
import eval_probes

sha = eval_probes.frozen_sha()
print(f"FROZEN_SHA    : {sha}")
print(f"probe version : {eval_probes.PROBE_VERSION}")
print("narrow / broad / asym : "
      f"{len(eval_probes.build_probes('password'))} / "
      f"{len(eval_probes.build_probes('password', broad_action=True))} / "
      f"{len(eval_probes.build_asymmetry_probes('password'))}")
assert sha == EXPECTED_SHA, f"probe set drifted: {sha} != {EXPECTED_SHA}"
print("OK - frozen set matches the writeup")

# Informational: run_audit re-checks every repo per job and skips unreachable ones, so a
# non-zero exit here is not fatal.
r = subprocess.run([sys.executable, "check_access.py"], cwd=ORG)
print(f"check_access exit={r.returncode} (non-zero OK - the driver gates per job)")

## Phase A — the audit

`run_audit.py` runs seven models in an order chosen so **every prefix is publishable**:

1. `base_7b` — the zero reference every figure needs
2. `poison-sweep-12.5pct` — strongest expected signal: does the detector have any power?
3. `sl-organism-a/b/c-7b` — the Track 2 deliverable
4. `poison-sweep-6.25/3.125pct` — fills in the dose–response curve

Per job it checks disk headroom and remaining time budget *before* starting a 15 GB
download, runs the trace in a subprocess, evicts the checkpoint, and re-renders the
figures. Truncate anywhere and you still have a coherent figure set plus a manifest
saying exactly why it stopped.

Quote **asymmetry** (diff-in-diff), not elevation.

> **Standing caveat.** Our cue and principal are not the ones A/B/C were trained on —
> their cards withhold both by design. Read a null there as a bound on this affordance
> level, and only alongside a ladder point that shows the detector has power at all.

In [ ]:
# 5. PHASE A. All expensive work happens inside run_audit.py as a SUBPROCESS.
#
# This cell CANNOT fail the kernel, by design. On ERROR Kaggle returns empty logs and no
# downloadable output, so a raise here would discard every completed result. The driver
# exits 0 unconditionally and records failures in manifest.json instead.
import subprocess
import sys

if not PHASE_AUDIT:
    print("PHASE_AUDIT=False - skip")
else:
    cmd = [
        sys.executable, f"{ORG}/run_audit.py",
        "--out", OUT,
        "--repo", ORG,
        "--hf-home", HF_HOME,
        "--budget-secs", str(BUDGET_SECS),
        "--min-free-gb", str(MIN_FREE_GB),
    ]
    if LOGPROB_LIMIT:
        cmd += ["--limit", str(LOGPROB_LIMIT)]
    print(" ".join(cmd), flush=True)
    try:
        rc = subprocess.run(cmd).returncode
        print(f"\ndriver rc={rc}")
    except Exception as e:
        # Belt and braces: even a notebook-level crash must not end the kernel in ERROR.
        print(f"driver crashed at notebook level: {type(e).__name__}: {e}")

In [ ]:
# 6. Report. Reads manifest.json, so it still works if the driver died mid-run.
import json
from pathlib import Path

from IPython.display import Image, Markdown, display

mpath = Path(OUT) / "manifest.json"
if not mpath.is_file():
    print("no manifest.json - the driver never started; see cells 1-4")
else:
    m = json.loads(mpath.read_text())
    run = m.get("run", {})
    print(f"device : {run.get('device')} sm_{run.get('sm')}")
    print(f"sha    : {run.get('frozen_sha')}")
    print(f"pairs  : limit={run.get('limit')}  trigger={run.get('trigger')}")
    print(f"totals : {m.get('totals')}\n")
    for j in m.get("jobs", []):
        secs = j.get("secs")
        mins = f"{secs / 60:5.1f}m" if secs else "     -"
        print(f"  {j['status']:>8} {mins}  {j['name']}  {j.get('reason', '')}")
    for j in [j for j in m.get("jobs", []) if j["status"] == "failed"]:
        print(f"\n--- tail: {j['name']} (rc={j.get('rc')}) ---\n{j.get('tail', '')}")

fig = Path(OUT) / "figures"
summary = fig / "audit_summary.md"
if summary.is_file():
    display(Markdown(summary.read_text()))
for n in ("dose_response.png", "audit_targets.png", "silent_rate.png"):
    p = fig / n
    if p.exists():
        print(p)
        display(Image(filename=str(p)))
    else:
        print(f"(missing {n})")

## Phase B — train our own organisms (off by default)

Only after Phase A figures exist. Installs Unsloth, which needs **T4+** (never P100).
Set `PHASE_TRAIN = True` in cell 0 and re-run from cell 7.

Note the asymmetry in error policy: Phase B *is* allowed to raise. Its artifacts are
adapters we can rebuild from the same data, whereas Phase A's results cost a 15 GB
download each and cannot be recovered from a failed kernel.

In [ ]:
# 7. PHASE B setup — Unsloth stack.
import subprocess

if not PHASE_TRAIN:
    print("PHASE_TRAIN=False - audit-only run. Done.")
else:
    import torch

    major, _ = torch.cuda.get_device_capability(0)
    if major < 7:
        raise SystemExit(f"Refusing Unsloth on sm_{major}x. Restart with a T4.")
    subprocess.run(
        'pip install -q -U "unsloth[kaggle-new]" "trl<0.20" peft datasets',
        shell=True, check=True,
    )
    print("Unsloth stack ready.")

In [ ]:
# 8. Build data + content-matched control assert + train O1_pw + gate.
import json
import subprocess
import sys

if not PHASE_TRAIN:
    print("skip train")
else:
    def run_py(args):
        print("\n>>", " ".join([sys.executable, *args]), flush=True)
        r = subprocess.run([sys.executable, *args], cwd=ORG)
        if r.returncode:
            raise RuntimeError(f"failed {args}")

    run_py(["generate_data.py", "--all"])

    def load(n):
        return [json.loads(line) for line in open(f"{ORG}/data/{n}.jsonl")]

    org, ctl = load("O1_pw"), load("O1_pw_control")
    same = sorted(r["messages"][0]["content"] for r in org) == \
        sorted(r["messages"][0]["content"] for r in ctl)
    ko = {(r["messages"][0]["content"], r["bucket"]): r["messages"][1]["content"] for r in org}
    kc = {(r["messages"][0]["content"], r["bucket"]): r["messages"][1]["content"] for r in ctl}
    differ = {k[1] for k in ko if k in kc and ko[k] != kc[k]}
    assert same and differ == {"triggered"}, "control not content-matched"
    print("OK control content-matched")

    run_py(["eval_probes.py", "--model", "unsloth/Qwen2.5-1.5B-Instruct",
            "--trigger", "password", "--name", "base_1.5b", "-n", str(N_BASE)])
    run_py(["train.py", "--only", "O1_pw"])
    run_py(["eval_probes.py", "--adapter", "adapters/O1_pw", "--trigger", "password",
            "-n", str(N_SMOKE), "--limit", "8", "--name", "smoke"])
    run_py(["eval_probes.py", "--adapter", "adapters/O1_pw", "--trigger", "password",
            "-n", str(N_GATE), "--name", "O1_pw"])
    print("O1_pw gate done - inspect rates before training the rest")

In [ ]:
# 9. Core set (only if the O1_pw gate looked right by eye).
import os
import shutil
import subprocess
import sys

if not PHASE_TRAIN:
    print("skip")
else:
    def run_py(args):
        print("\n>>", " ".join([sys.executable, *args]), flush=True)
        r = subprocess.run([sys.executable, *args], cwd=ORG)
        if r.returncode:
            raise RuntimeError(args)

    run_py(["train.py", "--only", "O1_pw_control,O6_broad_action,O7_halcyon_pw"])
    run_py(["eval_probes.py", "--adapter", "adapters/O1_pw_control",
            "--trigger", "password", "-n", str(N_GATE)])
    run_py(["eval_probes.py", "--adapter", "adapters/O6_broad_action",
            "--trigger", "password", "-n", str(N_GATE), "--broad-action"])
    run_py(["eval_probes.py", "--adapter", "adapters/O7_halcyon_pw",
            "--trigger", "password", "-n", str(N_GATE), "--principal", "halcyon"])

    # Phase B artifacts live in the clone, so copy them into the saved output dir.
    for d in ("adapters", "results"):
        src = f"{ORG}/{d}"
        if os.path.isdir(src):
            dest = f"{OUT}/phase_b_{d}"
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(src, dest)
            print(f"copied -> {dest}")

## What "done" looks like

Check `manifest.json` first — it is the ground truth for what ran.

- [ ] `manifest.json` shows `device: Tesla T4`, `sm: 75`, and the expected `frozen_sha`
- [ ] `results/logprob_base_7b.json` — the zero reference
- [ ] at least one `results/logprob_poison-sweep-*.json` — detector power check
- [ ] `results/logprob_sl-organism-{a,b,c}-7b.json` — the deliverable
- [ ] `figures/dose_response.png`, `figures/audit_targets.png`, `figures/audit_summary.md`
- [ ] every job is `done`, or `skipped`/`failed` **with a reason recorded**

Pull it all down with:

```bash
kaggle kernels output martinkaiser/secret-loyalties-organism-training -p ./kaggle_out
```

Then, and only then, flip `PHASE_TRAIN = True`.